# FEDS: Run server + clients (Colab)

Quick check: run FEDS server and MNIST clients. Artifacts are created in the next cell if missing. Server uses 300 rounds by default; interrupt after a few rounds for a quick check.

In [ ]:
# Clone or set path (Colab: clone; local: skip and set REPO)
import os
import sys

REPO = "path"

os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "utils"))
print("REPO:", REPO)

REPO: /content


In [ ]:
# Install deps from requirements.txt (run this first)
!pip install -q -r requirements.txt
import flwr
import torch
print("flwr:", flwr.__version__, " torch:", torch.__version__)

In [ ]:
# Create initial model and K pickle if missing
import subprocess
r = subprocess.run([sys.executable, os.path.join(REPO, "scripts", "create_artifacts_colab.py")], cwd=REPO, capture_output=True, text=True)
print(r.stdout or r.stderr or "OK")

In [ ]:
# Run server in background
import threading
import subprocess

env = os.environ.copy()
env["PYTHONPATH"] = os.path.join(REPO, "utils")
env["NUM_ROUNDS"] = "3"

def run_server():
    subprocess.run([sys.executable, os.path.join(REPO, "server.py")], cwd=REPO, env=env)

t = threading.Thread(target=run_server)
t.daemon = True
t.start()
import time
time.sleep(8)
print("Server started.")

In [ ]:
# Run 10 MNIST clients
client_script = os.path.join(REPO, "clients", "client-MNIST.py")
procs = []
for i in range(10):
    p = subprocess.Popen([sys.executable, client_script, "--seed", str(i)], cwd=REPO, env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    procs.append(p)
for p in procs:
    p.wait()
print("Clients finished.")

In [ ]:
# Check FEDS outputs
import json
k_tracker = os.path.join(REPO, "feds_k_tracker.json")
if os.path.exists(k_tracker):
    with open(k_tracker) as f:
        print(json.dumps(json.load(f), indent=2)[:1500])
else:
    print("No feds_k_tracker.json yet")
runs = os.path.join(REPO, "runs")
if os.path.isdir(runs):
    print("TensorBoard runs:", os.listdir(runs))